# Alur Lengkap Pemrosesan Data dari Foto Nota (OCR) hingga Penyimpanan Transaksi
**100% Sesuai Kode Sistem Asli**

Contoh gambar: `budis.jpg` di folder utama proyek (BOT).

---

In [1]:
# 1. Inisialisasi Environment dan Load Variabel
import os
import sys

# Tambahkan folder utama ke path agar bisa import module proyek
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

# Load variabel lingkungan dari .env
from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join('..', '.env'))

print("✅ Environment siap!")

✅ Environment siap!


---

In [2]:
# 2. Inisialisasi OCR Service (Hanya Mistral, No Paddle)
from services.ocr_service import OCRService

ocr_service = OCRService()
if not ocr_service.load_model():
    print(f"⚠️ OCR gagal dimuat: {ocr_service._disabled_reason}")
else:
    print("✅ OCR Service siap!")

✅ OCR Service siap!


---

In [3]:
# 3. Baca Gambar dan Jalankan OCR
# Contoh gambar: budis.jpg di folder utama
gambar_path = os.path.join('..', 'budis.jpg')

if not os.path.exists(gambar_path):
    print(f"⚠️ File gambar tidak ditemukan: {gambar_path}")
    print("Silakan letakkan gambar Anda dengan nama 'budis.jpg' di folder utama proyek, atau ubah variabel 'gambar_path' di atas.")
else:
    print(f"📂 Membaca gambar dari: {gambar_path}")
    teks_ocr = ocr_service.extract_text(gambar_path)
    
    if teks_ocr.startswith("Error OCR:"):
        print(f"❌ {teks_ocr}")
    else:
        print("\n📝 Hasil OCR:")
        print("=" * 80)
        print(teks_ocr)
        print("=" * 80)

📂 Membaca gambar dari: ..\budis.jpg

📝 Hasil OCR:
19-06-2024

Nama : Budi budis

bembeng   100 ctn
meses     20 ctn
twilo    100 ctn


---

In [4]:
# 4. Load Master Data (Barang & Metode Pembayaran) dari Database
from core.master_data import get_all_barang, get_all_metode, muat_metode_keywords

print("🔄 Memuat master data dari database...")

# Load master barang
semua_barang = get_all_barang()
daftar_nama_barang = [b["nama"] for b in semua_barang]

# Load master metode pembayaran
semua_metode = get_all_metode()
mapping_metode = muat_metode_keywords()

print(f"✅ Loaded: {len(semua_barang)} barang, {len(semua_metode)} metode pembayaran")
if semua_barang:
    print(f"   Contoh barang: {', '.join([b['nama'] for b in semua_barang[:3]])}")

🔄 Memuat master data dari database...
✅ Loaded: 13 barang, 0 metode pembayaran
   Contoh barang: Willo, bembeng, Adangrow


---

In [5]:
# 5A. Demonstrasi Preprocessing Tahap 1: Remove Bot UI Noise
import re

# Regex dari production code
_BOT_UI_NOISE_RE = re.compile(
    r"(?:"
    r"hasil\s*ekstraksi\s*ocr"
    r"|rangkuman\s*ekstraksi\s*mesin"
    r"|data\s*belum\s*lengkap"
    r"|lengkapi\s*data"
    r"|kirim\s*data"
    r"|edit\s*kembali"
    r"|data\s*belum\s*masuk\s*database"
    r")",
    flags=re.IGNORECASE,
)

_BOT_UI_LABEL_LINE_RE = re.compile(
    r"^(?:📅|👤|⚙️|📦|🔢|📋|💵|💳|🏦|💸|⚠️)\s*",
    flags=re.IGNORECASE,
)

def _normalize_ocr_line(text):
    line = (text or "").strip()
    line = re.sub(r"^[\s👤📅📦🔢💳🏦💵⚠️✅🤖]+", "", line).strip()
    line = re.sub(r"^\[\d+\]\s*", "", line).strip()
    line = re.sub(r"^[\s~•*\-–—]+", "", line).strip()
    line = re.sub(r"\s+", " ", line).strip()
    return line

def _remove_bot_ui_noise(text):
    if not text:
        return text
    out_lines = []
    for raw in str(text).replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        line = _normalize_ocr_line(raw)
        if not line:
            continue
        if _BOT_UI_NOISE_RE.search(line):
            continue
        if _BOT_UI_LABEL_LINE_RE.match(line) and ":" in line:
            continue
        out_lines.append(line)
    return "\n".join(out_lines)

# Contoh OCR dengan noise UI
ocr_mentah = """🔍 HASIL EKSTRAKSI OCR (1/1)
👤 Pelanggan: Pak Andi
📅 Tanggal: 29-06-2025
Willo 10 toples
Bembeng 5 pack
Total: Rp 250.000
Metode: Transfer Lunas
⚠️ Data belum masuk database"""

print("=== OCR MENTAH (dengan noise) ===")
print(ocr_mentah)

ocr_bersih = _remove_bot_ui_noise(ocr_mentah)
print("\n=== SETELAH REMOVE UI NOISE ===")
print(ocr_bersih)
print("\n")


=== OCR MENTAH (dengan noise) ===
🔍 HASIL EKSTRAKSI OCR (1/1)
👤 Pelanggan: Pak Andi
📅 Tanggal: 29-06-2025
Willo 10 toples
Bembeng 5 pack
Total: Rp 250.000
Metode: Transfer Lunas
⚠️ Data belum masuk database

=== SETELAH REMOVE UI NOISE ===
Pelanggan: Pak Andi
Tanggal: 29-06-2025
Willo 10 toples
Bembeng 5 pack
Total: Rp 250.000
Metode: Transfer Lunas




In [6]:
# 5B. Demonstrasi Preprocessing Tahap 2: Structured Parsing
# Contoh dari handlers/photo_handler.py

def _split_ocr_blocks(text):
    """Pisahkan OCR menjadi blocks berdasarkan tanggal"""
    from datetime import datetime
    blocks = []
    current_block = {"date": None, "lines": []}
    date_pattern = re.compile(r"(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})")
    
    for line in (text or "").split("\n"):
        line = line.strip()
        if not line:
            continue
        
        # Cari tanggal di awal baris
        m = date_pattern.search(line)
        if m:
            if current_block["lines"]:
                blocks.append(current_block)
            current_block = {"date": m.group(), "lines": [line]}
        else:
            current_block["lines"].append(line)
    
    if current_block["lines"]:
        blocks.append(current_block)
    return blocks

def _extract_block_name(lines, daftar_b=None):
    """Extract nama pelanggan dari block"""
    for line in lines:
        # Cari pattern: "nama xxx" atau "kepada xxx"
        if "nama" in line.lower():
            parts = re.split(r"[:\s]+", line.lower())
            idx = parts.index("nama") if "nama" in parts else -1
            if idx >= 0 and idx + 1 < len(parts):
                return parts[idx + 1].title()
        if "kepada" in line.lower():
            parts = re.split(r"[:\s]+", line.lower())
            idx = parts.index("kepada") if "kepada" in parts else -1
            if idx >= 0 and idx + 1 < len(parts):
                return parts[idx + 1].title()
    return None

def _extract_item_lines_from_block(lines, daftar_b=None):
    """Extract item barang (qty + satuan pattern) dari block"""
    items = []
    for line in lines:
        # Cari pattern: "barang_name qty satuan"
        m = re.search(r"(.+?)\s+(\d+)\s+(toples|pack|dus|karton|pcs|box)\b", line, re.IGNORECASE)
        if m:
            items.append(f"{m.group(1)} {m.group(2)} {m.group(3)}")
    return items

# Demonstrasi
print("=== HASIL PARSING TERSTRUKTUR ===\n")
blocks = _split_ocr_blocks(ocr_bersih)

for block_idx, block in enumerate(blocks, 1):
    print(f"Block #{block_idx}:")
    print(f"  Tanggal: {block['date']}")
    
    nama = _extract_block_name(block["lines"])
    print(f"  Nama: {nama}")
    
    items = _extract_item_lines_from_block(block["lines"])
    print(f"  Items:")
    for item in items:
        print(f"    - {item}")
    print()

print("✅ Parsing terstruktur selesai")
print("   Jika parsing BERHASIL: hasil langsung (Path A - cepat)")
print("   Jika parsing GAGAL: lanjut ke NLP biasa (Path B - detail)")
print("\n")


=== HASIL PARSING TERSTRUKTUR ===

Block #1:
  Tanggal: None
  Nama: None
  Items:

Block #2:
  Tanggal: 29-06-2025
  Nama: None
  Items:
    - Willo 10 toples
    - Bembeng 5 pack

✅ Parsing terstruktur selesai
   Jika parsing BERHASIL: hasil langsung (Path A - cepat)
   Jika parsing GAGAL: lanjut ke NLP biasa (Path B - detail)




## ⚠️ Penjelasan: Alur OCR → NLP (Sebenarnya 100% Sesuai Kode)

**Input NLP TIDAK murni dari OCR langsung!** Ada 3 tahap preprocessing:

### Alur Sebenarnya:
```
1. OCR Extraction (Mistral API)
   ↓
2. _remove_bot_ui_noise()
   → Hapus emoji, label bot, indeks numbering
   ↓
3. _build_structured_nlp_input()
   → Parse OCR menjadi blocks (berdasarkan tanggal, nama)
   → Extract payment entries, item lines, metode pembayaran
   ↓
   ├─ BERHASIL extract terstruktur?
   │  └─ _build_sales_results_from_ocr_entries()
   │     → Parse items dengan regex patterns
   │     → Generate entitas langsung (SKIP proses_nlp)
   │     → **Hasil langsung tanpa NLP!**
   │
   └─ GAGAL extract terstruktur?
      └─ koreksi_teks() + proses_nlp(already_normalized=True)
         → Proses sebagai teks natural language
         → Fallback ke NLP biasa
```

### Perbedaan 2 Path:
- **Path A (Structured)**: OCR → Preprocessing → Regex Parsing → Hasil (CEPAT)
- **Path B (NLP Fallback)**: OCR → Preprocessing → Text Normalization → NLP → Hasil (DETAIL)

Sistem memilih Path A jika OCR sudah terstruktur dengan baik, Path B jika OCR ambigu/natural language.


In [7]:
# 5C. Pilih Path: Terstruktur vs NLP Fallback
from nlp.processor import proses_nlp
from nlp.normalizer import koreksi_teks

# Kita gunakan hasil OCR bersih dari cell sebelumnya
if 'ocr_bersih' not in locals():
    ocr_bersih = """Tanggal: 29-06-2025
Kepada: Pak Budi
Willo 10 toples
Bembeng 5 pack
Transfer Lunas"""

print("=" * 80)
print("DEMONSTRASI: 2 PATH PEMROSESAN")
print("=" * 80)

# Check apakah parsing terstruktur berhasil
blocks = _split_ocr_blocks(ocr_bersih)
has_structured_data = all(_extract_block_name(block["lines"]) and _extract_item_lines_from_block(block["lines"]) for block in blocks)

print(f"\n📊 Hasil Parsing Terstruktur:")
print(f"   - Blocks ditemukan: {len(blocks)}")
print(f"   - Parsing berhasil: {has_structured_data}")

if has_structured_data:
    print("\n✅ PATH A: STRUCTURED EXTRACTION (Cepat)")
    print("   OCR sudah terstruktur → Parse dengan regex → Hasil langsung")
    print("   (Bypass NLP, lebih cepat & predictable)\n")
    
    nlp_results = []
    for block in blocks:
        nama = _extract_block_name(block["lines"]) or "Unknown"
        tgl = block["date"] or "29-06-2025"
        items = _extract_item_lines_from_block(block["lines"])
        
        for item in items:
            entitas = {
                "TANGGAL": tgl,
                "NAMA": nama,
                "AKSI": "Tambah Penjualan",
                "BARANG": item.split()[0],
                "JUMLAH": " ".join(item.split()[1:]),
                "STATUS": "Belum Lunas",
                "METODE_PEMBAYARAN": "Transfer",
            }
            nlp_results.append({"intent": "Catat_Penjualan_Cicil", "entitas": entitas})
    
else:
    print("\n⚠️ PATH B: NLP FALLBACK (Detail)")
    print("   OCR tidak terstruktur → Normalisasi teks → Proses dengan NLP\n")
    
    # Normalisasi teks dulu
    teks_norm = koreksi_teks(ocr_bersih, daftar_barang=daftar_nama_barang)
    
    # Baru proses dengan NLP
    nlp_results = proses_nlp(
        teks_norm,
        db_metode=semua_metode,
        daftar_barang=daftar_nama_barang,
        mapping_metode=mapping_metode,
        already_normalized=True
    )

# Tampilkan hasil dari kedua path
print("\n" + "=" * 80)
print("HASIL AKHIR (dari salah satu path di atas):")
print("=" * 80 + "\n")

if nlp_results:
    print(f"✅ Berhasil mengekstrak {len(nlp_results)} transaksi:\n")
    for idx, transaksi in enumerate(nlp_results, 1):
        print(f"Transaksi #{idx}:")
        entitas = transaksi.get("entitas", {}) or {}
        for key, value in entitas.items():
            if value is not None:
                print(f"  {key}: {value}")
        print()
else:
    print("❌ Tidak ada transaksi yang bisa diekstrak.")


DEMONSTRASI: 2 PATH PEMROSESAN

📊 Hasil Parsing Terstruktur:
   - Blocks ditemukan: 2
   - Parsing berhasil: False

⚠️ PATH B: NLP FALLBACK (Detail)
   OCR tidak terstruktur → Normalisasi teks → Proses dengan NLP


HASIL AKHIR (dari salah satu path di atas):

✅ Berhasil mengekstrak 1 transaksi:

Transaksi #1:
  NAMA: Pak Andi
  AKSI: Tambah Penjualan
  BARANG: Willo
  JUMLAH: 10 toples
  SATUAN: toples
  STATUS: Lunas
  METODE_PEMBAYARAN: Transfer
  SEMUA: False



## 📋 Ringkasan: Alur Preprocessing OCR → NLP di Production

### Lokasi Kode di Production:
- **OCR Extraction**: `services/ocr_service.py` → `extract_text()`
- **Noise Removal**: `handlers/photo_handler.py` → `_remove_bot_ui_noise()`
- **Structural Parsing**: `handlers/photo_handler.py` → `_build_structured_nlp_input()` + `_build_sales_results_from_ocr_entries()`
- **NLP Fallback**: `handlers/photo_handler.py` → `koreksi_teks()` + `proses_nlp()`

### Alur di `handlers/photo_handler.py` (Line 894-922):
```python
# Step 1: Remove noise
clean_ocr_text = _remove_bot_ui_noise(ocr_text)

# Step 2: Try structured parsing
nlp_input_text, payment_entries, sales_contexts, sales_entries = \
    _build_structured_nlp_input(clean_ocr_text, daftar_b=daftar_b)

# Step 3: Pilih path
if sales_entries:
    # Path A: Structured extraction berhasil
    results_nlp = _build_sales_results_from_ocr_entries(sales_entries, ...)
else:
    # Path B: Fallback ke NLP
    user_text_norm = koreksi_teks(nlp_input_text, daftar_barang=daftar_b)
    results_nlp = proses_nlp(user_text_norm, already_normalized=True, ...)
```

### Kesimpulan:
- ✅ **Input NLP BUKAN murni dari OCR** → Ada 3 tahap preprocessing
- ✅ **Path A (Structured)** lebih diprioritaskan jika OCR terstruktur (tanggal, nama, item jelas)
- ✅ **Path B (NLP)** hanya jika Path A gagal (OCR natural language atau ambigu)
- ✅ **Keduanya menghasilkan format entitas sama** untuk tahap berikutnya (perhitungan, penyimpanan)


---

In [8]:
# 6. (Opsional) Hitung Total Harga jika master barang tersedia
from core.master_data import cari_harga_default, parse_rupiah, format_rupiah

if 'nlp_results' in locals() and nlp_results and semua_barang:
    print("💵 Menghitung total harga transaksi...")
    print("-" * 80)
    
    for idx, transaksi in enumerate(nlp_results, 1):
        entitas = transaksi.get("entitas", {}) or {}
        nama_barang = entitas.get("BARANG")
        satuan = entitas.get("SATUAN")
        jumlah_str = entitas.get("JUMLAH")
        
        # Cari harga
        harga_list = cari_harga_default(None, nama_barang, satuan_cari=satuan, semua_barang=semua_barang) if nama_barang else []
        harga_satuan = harga_list[0]["harga"] if harga_list else 0
        
        # Hitung jumlah
        import re
        angka_jumlah_match = re.search(r"\d+", str(jumlah_str)) if jumlah_str else None
        jumlah = int(angka_jumlah_match.group()) if angka_jumlah_match else 1
        
        # Hitung total
        total = harga_satuan * jumlah
        
        print(f"Transaksi #{idx}:")
        print(f"  Barang: {nama_barang}")
        print(f"  Jumlah: {jumlah} {satuan}")
        print(f"  Harga Satuan: {format_rupiah(harga_satuan)}")
        print(f"  Total: {format_rupiah(total)}")
        print("-" * 80)
else:
    print("ℹ️ Tidak bisa menghitung harga karena transaksi tidak tersedia atau master barang kosong.")


💵 Menghitung total harga transaksi...
--------------------------------------------------------------------------------
Transaksi #1:
  Barang: Willo
  Jumlah: 10 toples
  Harga Satuan: Rp 504.000
  Total: Rp 5.040.000
--------------------------------------------------------------------------------
